# 04. AI 반주자와 함께 연주하다 — **협업**

## 이 노트북의 위치

```
기반     : NB01 (듣다)
확장     : NB02 (시각화)
협업 ✦   : NB04 (AI 반주자) ← 여기
통합     : NB05 (무대)
```

**확장**은 연주자가 만든 데이터를 새 차원(영상)으로 바꾸는 **일방향** 작업이었습니다.  
**협업**은 다릅니다 — AI가 내 연주를 **실시간으로 따라가고**, 거기에 맞춰 **반주까지 생성**합니다.

### 이 노트북에서 체험하는 두 가지 데모

| # | 데모 | 핵심 기술 |
|---|------|----------|
| **A** | **오프라인 AI 반주** | matchmaker 정렬 → 타이밍 변형 → FluidSynth 합성 → 믹스 |
| **B** | **라이브 AI 반주** | 서버가 실시간으로 연주를 추적하며 반주를 스트리밍 |

### matchmaker

**[matchmaker](https://github.com/pymatchmaker/matchmaker)**는 ISMIR 2025에서 발표된 오픈소스 악보 추적(score following) 라이브러리입니다.

- 악보(MIDI)와 연주(오디오)를 받으면, **"지금 악보의 몇 번째 박자를 치고 있는가"**를 실시간으로 알려줍니다.
- 루바토가 있어도, 빨라져도, 느려져도 따라갑니다.
- 이 위치 정보로 **반주 파트의 타이밍을 변형**하면 → AI 반주자가 완성됩니다.

**입력**: `assets/scores/*.mid` (원본 악보), `assets/*.wav` (연주 녹음)  
**출력**: `artifacts/alignment/*.json`, `artifacts/accompaniment/*.wav`

In [ ]:
# Colab용 설치 (서버 시연 환경에서는 건너뛰어도 됩니다)
!apt-get -qq install -y fluidsynth portaudio19-dev > /dev/null 2>&1
!pip install -q pymatchmaker pretty_midi midi2audio numpy matplotlib librosa soundfile

import warnings
warnings.filterwarnings('ignore')
print('✅ 설치 완료!')

---
## 1. 악보 추적이란?

피아니스트가 악보를 보며 연주합니다.  
옆에서 듣는 사람(또는 AI)이 **"지금 악보의 어디를 치고 있는지"** 실시간으로 파악하는 것 — 이것이 **악보 추적(Score Following)**입니다.

```
연주 오디오  ──▶  matchmaker  ──▶  "지금 악보의 17.3번째 박자"
     +                              ↓
악보 MIDI   ──▶                 시각화/페이지 넘기기/자동 반주
```

### 왜 어려운가?

- 메트로놈에 맞춰 치면 쉽겠지만, 실제 연주에는 **루바토**, **급가속**, **일시 정지**가 있습니다.
- Satie의 느린 루바토와 Prokofiev의 빠른 토카타 — AI가 둘 다 따라갈 수 있을까?
- 이것을 직접 확인하는 것이 이 노트북의 핵심입니다.

---
## 2. 오프라인 정렬 — 녹음으로 테스트

라이브 마이크 없이, **녹음 파일**을 악보와 정렬합니다.  
matchmaker에 `performance_file`을 넘기면 녹음을 실시간처럼 재생하며 정렬합니다.

In [ ]:
from pathlib import Path
import json
import numpy as np

ARTIFACTS = Path('artifacts')
ALIGNMENT_DIR = ARTIFACTS / 'alignment'
ALIGNMENT_DIR.mkdir(parents=True, exist_ok=True)

pieces = {
    'satie': {
        'title': 'Satie — Gymnopédie No.1',
        'score': 'assets/scores/satie_gymnopedie.mid',
        'audio': 'assets/satie_gymnopedie_no1.wav',
    },
    'prokofiev': {
        'title': 'Prokofiev — Toccata Op.11',
        'score': 'assets/scores/prokofiev_toccata.mid',
        'audio': 'assets/prokofiev_toccata_op11.wav',
    },
}

for key, p in pieces.items():
    for f in ['score', 'audio']:
        if not Path(p[f]).exists():
            raise FileNotFoundError(f"{p[f]} 없음")
    print(f"✅ {p['title']}")

print(f"\n정렬 결과 저장 위치: {ALIGNMENT_DIR}")

In [ ]:
from matchmaker import Matchmaker
import time

print("🎹 Satie — Gymnopédie No.1 정렬 시작...")
t0 = time.time()

mm_satie = Matchmaker(
    score_file=pieces['satie']['score'],
    performance_file=pieces['satie']['audio'],
    input_type="audio",
)

satie_positions = []
for pos in mm_satie.run():
    satie_positions.append(float(pos))

elapsed = time.time() - t0
print(f"✅ 완료! {len(satie_positions)}개 프레임, {elapsed:.1f}초 소요")
print(f"   악보 위치 범위: {min(satie_positions):.1f} ~ {max(satie_positions):.1f} beats")

In [ ]:
print("🎹 Prokofiev — Toccata Op.11 정렬 시작...")
t0 = time.time()

mm_prok = Matchmaker(
    score_file=pieces['prokofiev']['score'],
    performance_file=pieces['prokofiev']['audio'],
    input_type="audio",
)

prok_positions = []
for pos in mm_prok.run():
    prok_positions.append(float(pos))

elapsed = time.time() - t0
print(f"✅ 완료! {len(prok_positions)}개 프레임, {elapsed:.1f}초 소요")
print(f"   악보 위치 범위: {min(prok_positions):.1f} ~ {max(prok_positions):.1f} beats")

---
## 3. 정렬 결과 시각화

X축은 **연주 시간**(초), Y축은 **악보 위치**(beat).  
완벽한 정렬은 매끄러운 상승 곡선입니다. 루바토가 있으면 곡선이 늘어지고, 빠른 구간에서는 급경사가 됩니다.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (key, positions) in zip(axes, [('satie', satie_positions), ('prokofiev', prok_positions)]):
    n = len(positions)
    frame_rate = mm_satie.frame_rate if key == 'satie' else mm_prok.frame_rate
    times = np.arange(n) / frame_rate
    
    ax.plot(times, positions, linewidth=0.8, color='#4F46E5' if key == 'satie' else '#DC2626')
    ax.set_xlabel('연주 시간 (초)', fontsize=11)
    ax.set_ylabel('악보 위치 (beat)', fontsize=11)
    ax.set_title(pieces[key]['title'], fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(ALIGNMENT_DIR / 'alignment_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("💾 alignment_curves.png 저장됨")

---
## 4. 두 곡 비교 — 정렬 난이도

같은 알고리즘이 두 곡을 어떻게 다르게 따라가는지 분석합니다.

In [ ]:
def analyze_alignment(positions, frame_rate):
    """정렬 곡선의 특성을 계산합니다."""
    pos = np.array(positions)
    velocity = np.diff(pos) * frame_rate  # beats/sec
    return {
        'total_frames': len(pos),
        'duration_sec': len(pos) / frame_rate,
        'beat_range': (float(pos.min()), float(pos.max())),
        'avg_tempo_bps': float(np.mean(velocity)),
        'tempo_std_bps': float(np.std(velocity)),
        'tempo_cv': float(np.std(velocity) / max(np.mean(velocity), 1e-6)),
        'max_speed_bps': float(np.max(np.abs(velocity))),
    }

fr_s = mm_satie.frame_rate
fr_p = mm_prok.frame_rate

stats_s = analyze_alignment(satie_positions, fr_s)
stats_p = analyze_alignment(prok_positions, fr_p)

print(f"{'':20s} {'Satie':>12s} {'Prokofiev':>12s}")
print(f"{'─'*46}")
print(f"{'연주 길이 (초)':20s} {stats_s['duration_sec']:12.1f} {stats_p['duration_sec']:12.1f}")
print(f"{'평균 속도 (beats/s)':20s} {stats_s['avg_tempo_bps']:12.2f} {stats_p['avg_tempo_bps']:12.2f}")
print(f"{'속도 편차 (σ)':20s} {stats_s['tempo_std_bps']:12.2f} {stats_p['tempo_std_bps']:12.2f}")
print(f"{'속도 변동계수 (CV)':20s} {stats_s['tempo_cv']:12.2f} {stats_p['tempo_cv']:12.2f}")
print(f"{'최대 순간 속도':20s} {stats_s['max_speed_bps']:12.2f} {stats_p['max_speed_bps']:12.2f}")
print()
if stats_s['tempo_cv'] > stats_p['tempo_cv']:
    print("→ Satie의 속도 변동이 더 큽니다 — 루바토가 많다는 뜻입니다.")
else:
    print("→ Prokofiev의 속도 변동이 더 큽니다 — 빠르고 불규칙한 구간이 있다는 뜻입니다.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (key, positions, fr) in zip(axes, [
    ('satie', satie_positions, fr_s),
    ('prokofiev', prok_positions, fr_p),
]):
    pos = np.array(positions)
    velocity = np.diff(pos) * fr
    window = min(50, len(velocity) // 4)
    if window > 0:
        smooth = np.convolve(velocity, np.ones(window)/window, mode='valid')
        times = np.arange(len(smooth)) / fr
    else:
        smooth = velocity
        times = np.arange(len(velocity)) / fr
    
    color = '#4F46E5' if key == 'satie' else '#DC2626'
    ax.plot(times, smooth, linewidth=0.8, color=color)
    ax.axhline(y=np.mean(velocity), color=color, linestyle='--', alpha=0.5, label=f'평균 {np.mean(velocity):.2f} bps')
    ax.set_xlabel('연주 시간 (초)', fontsize=11)
    ax.set_ylabel('추적 속도 (beats/sec)', fontsize=11)
    ax.set_title(f'{pieces[key]["title"]} — 속도 변화', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(ALIGNMENT_DIR / 'tempo_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("💾 tempo_curves.png 저장됨")

### 핵심 질문

- **Satie**: 루바토 구간에서 AI가 잘 따라갔나요? 곡선이 부드럽게 늘어지면 성공입니다.
- **Prokofiev**: 빠른 패시지에서 AI가 놓치지 않았나요? 급경사 구간에서 끊김이 없으면 성공입니다.
- 어느 곡이 AI에게 더 어려웠을까요? 속도 변동계수(CV)가 힌트입니다.

---
## 데모 A: 오프라인 AI 반주

Satie의 Gymnopédie No.1은 **오른손(멜로디)**과 **왼손(반주)**으로 나뉩니다.

시나리오:
1. 피아니스트가 **멜로디만** 녹음합니다 (실제로는 전체 녹음을 사용)
2. matchmaker가 녹음과 악보를 **정렬** → "이 시점에서 악보의 몇 번째 박자인지" 파악
3. 정렬 정보로 **반주 파트의 타이밍을 변형** → 연주자의 루바토에 맞춤
4. FluidSynth로 **변형된 반주를 합성**
5. 원본 녹음 + AI 반주를 **믹스** → 들어봅니다

```
녹음 ──▶ matchmaker ──▶ 정렬(beat↔time) ──▶ 반주 타이밍 변형 ──▶ FluidSynth ──▶ 믹스
  +                                              ↑
악보 MIDI ─────────── 멜로디/반주 분리 ──────────┘
```

### 5-1. 악보를 멜로디와 반주로 분리

Satie Gymnopédie의 MIDI 파일은 이미 **treble**(오른손)과 **bass**(왼손)로 나뉘어 있습니다.

In [ ]:
import pretty_midi

score = pretty_midi.PrettyMIDI(pieces['satie']['score'])

melody_inst = score.instruments[0]   # treble (오른손)
accomp_inst = score.instruments[1]   # bass   (왼손)

print(f"🎵 멜로디 (오른손): {len(melody_inst.notes)}개 음표, "
      f"음역 {min(n.pitch for n in melody_inst.notes)}~{max(n.pitch for n in melody_inst.notes)}")
print(f"🎹 반주   (왼손):   {len(accomp_inst.notes)}개 음표, "
      f"음역 {min(n.pitch for n in accomp_inst.notes)}~{max(n.pitch for n in accomp_inst.notes)}")
print(f"\n악보 길이: {score.get_end_time():.1f}초 (템포 60 BPM → 1 beat = 1초)")

### 5-2. 정렬 결과로 타이밍 매핑 만들기

matchmaker가 알려준 정렬: **연주 프레임 → 악보 beat**  
우리에게 필요한 것: **악보 시간 → 연주 시간** (반주 타이밍을 변형하려면 역방향이 필요)

Satie의 템포가 60 BPM이므로 **악보 beat = 악보 시간(초)**입니다.

In [ ]:
def build_score_to_perf_map(positions, frame_rate):
    """matchmaker 정렬 결과에서 악보시간→연주시간 매핑을 구축한다.
    
    positions: matchmaker가 출력한 score beat 배열 (프레임별)
    frame_rate: 초당 프레임 수
    
    Returns: (score_beats, perf_times) — 단조증가하는 매핑 테이블
    """
    perf_times_all = np.arange(len(positions)) / frame_rate
    score_beats_all = np.array(positions)
    
    # matchmaker 출력은 단조증가가 아닐 수 있으므로 정리
    # 단조증가하는 부분만 추출
    mask = np.ones(len(score_beats_all), dtype=bool)
    max_so_far = -np.inf
    for i in range(len(score_beats_all)):
        if score_beats_all[i] > max_so_far:
            max_so_far = score_beats_all[i]
        else:
            mask[i] = False
    
    score_beats = score_beats_all[mask]
    perf_times = perf_times_all[mask]
    
    print(f"매핑 테이블: {len(score_beats)}개 포인트")
    print(f"  악보 범위: {score_beats[0]:.1f} ~ {score_beats[-1]:.1f} beats")
    print(f"  연주 범위: {perf_times[0]:.1f} ~ {perf_times[-1]:.1f} 초")
    return score_beats, perf_times

score_beats_map, perf_times_map = build_score_to_perf_map(satie_positions, fr_s)

### 5-3. 반주 타이밍 변형 (Time-Warping)

반주의 각 음표가 악보에서 시작/끝나는 시간을 → 연주자의 실제 타이밍으로 바꿉니다.  
연주자가 느리게 친 구간에서는 반주도 느려지고, 빠르게 친 구간에서는 반주도 빨라집니다.

In [ ]:
def score_time_to_perf_time(t_score, score_beats, perf_times):
    """악보 시간을 연주 시간으로 변환 (선형 보간)."""
    return float(np.interp(t_score, score_beats, perf_times))

def warp_accompaniment(accomp_notes, score_beats, perf_times):
    """반주 음표들의 타이밍을 연주자 타이밍에 맞게 변형한다."""
    warped = []
    for note in accomp_notes:
        new_start = score_time_to_perf_time(note.start, score_beats, perf_times)
        new_end = score_time_to_perf_time(note.end, score_beats, perf_times)
        if new_end <= new_start:
            new_end = new_start + 0.05
        warped_note = pretty_midi.Note(
            velocity=note.velocity,
            pitch=note.pitch,
            start=new_start,
            end=new_end,
        )
        warped.append(warped_note)
    return warped

warped_notes = warp_accompaniment(accomp_inst.notes, score_beats_map, perf_times_map)

# 변형 전후 비교
print("타이밍 변형 전후 비교 (처음 5개 음표):")
print(f"{'원본 시작':>10s} {'원본 끝':>10s} │ {'변형 시작':>10s} {'변형 끝':>10s} │ {'시간 차이':>10s}")
print("─" * 60)
for orig, warped in zip(accomp_inst.notes[:5], warped_notes[:5]):
    diff = warped.start - orig.start
    print(f"{orig.start:10.2f} {orig.end:10.2f} │ {warped.start:10.2f} {warped.end:10.2f} │ {diff:+10.2f}s")
print(f"\n✅ {len(warped_notes)}개 반주 음표 타이밍 변형 완료")

### 5-4. 변형된 반주를 시각화

원본 반주와 변형 반주의 타이밍을 피아노롤로 비교합니다.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

for ax, notes, label, color in [
    (axes[0], accomp_inst.notes, '원본 반주 (악보 타이밍)', '#6366F1'),
    (axes[1], warped_notes, 'AI 반주 (연주자 타이밍에 맞춤)', '#059669'),
]:
    for n in notes:
        ax.barh(n.pitch, n.end - n.start, left=n.start, height=0.7,
                color=color, alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_ylabel('MIDI Pitch')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.2, axis='x')

axes[1].set_xlabel('시간 (초)')
plt.tight_layout()
plt.savefig(str(ALIGNMENT_DIR / 'accompaniment_warp.png'), dpi=150, bbox_inches='tight')
plt.show()
print("💾 accompaniment_warp.png 저장됨")

### 5-5. FluidSynth로 반주 합성

변형된 반주 MIDI를 FluidSynth로 합성해서 오디오로 만듭니다.

In [ ]:
from midi2audio import FluidSynth

ACCOMP_DIR = ARTIFACTS / 'accompaniment'
ACCOMP_DIR.mkdir(parents=True, exist_ok=True)

# 변형된 반주를 MIDI 파일로 저장
warped_midi = pretty_midi.PrettyMIDI(initial_tempo=60.0)
piano = pretty_midi.Instrument(program=0, name='AI Accompaniment')
piano.notes = warped_notes
warped_midi.instruments.append(piano)

warped_midi_path = str(ACCOMP_DIR / 'satie_accomp_warped.mid')
warped_midi.write(warped_midi_path)
print(f"💾 변형 반주 MIDI: {warped_midi_path}")

# FluidSynth로 합성
SF2_PATH = '/usr/share/sounds/sf2/FluidR3_GM.sf2'
fs = FluidSynth(SF2_PATH, sample_rate=44100)

warped_wav_path = str(ACCOMP_DIR / 'satie_accomp_warped.wav')
fs.midi_to_audio(warped_midi_path, warped_wav_path)
print(f"🔊 AI 반주 오디오: {warped_wav_path}")

# 원본 반주도 합성 (비교용)
orig_midi = pretty_midi.PrettyMIDI(initial_tempo=60.0)
orig_piano = pretty_midi.Instrument(program=0, name='Original Accompaniment')
orig_piano.notes = list(accomp_inst.notes)
orig_midi.instruments.append(orig_piano)

orig_midi_path = str(ACCOMP_DIR / 'satie_accomp_original.mid')
orig_midi.write(orig_midi_path)
orig_wav_path = str(ACCOMP_DIR / 'satie_accomp_original.wav')
fs.midi_to_audio(orig_midi_path, orig_wav_path)
print(f"🔊 원본 반주 오디오: {orig_wav_path}")

### 5-6. 원본 녹음 + AI 반주 믹스

연주자의 녹음(전체) 위에 AI가 생성한 반주를 겹칩니다.  
두 버전을 비교해 봅시다:
- **A**: 녹음 + 원본 반주 (메트로놈 타이밍 — 연주자와 안 맞음)
- **B**: 녹음 + AI 반주 (연주자 타이밍에 맞춤 — AI 반주자)

In [ ]:
import librosa
import soundfile as sf

# 원본 녹음 로드
recording, sr = librosa.load(pieces['satie']['audio'], sr=44100, mono=True)
print(f"🎤 원본 녹음: {len(recording)/sr:.1f}초, sr={sr}")

# AI 반주 로드
accomp_warped, _ = librosa.load(warped_wav_path, sr=44100, mono=True)
accomp_original, _ = librosa.load(orig_wav_path, sr=44100, mono=True)

def mix_audio(track_a, track_b, gain_b=0.6):
    """두 오디오를 믹스한다. 길이가 다르면 짧은 쪽에 패딩."""
    max_len = max(len(track_a), len(track_b))
    a = np.pad(track_a, (0, max_len - len(track_a)))
    b = np.pad(track_b, (0, max_len - len(track_b)))
    mixed = a + b * gain_b
    mixed = mixed / np.max(np.abs(mixed) + 1e-8)  # normalize
    return mixed

# 믹스 A: 녹음 + 원본 반주 (안 맞는 버전)
mix_orig = mix_audio(recording, accomp_original, gain_b=0.5)
mix_orig_path = str(ACCOMP_DIR / 'satie_mix_original_accomp.wav')
sf.write(mix_orig_path, mix_orig, sr)

# 믹스 B: 녹음 + AI 반주 (맞는 버전)
mix_ai = mix_audio(recording, accomp_warped, gain_b=0.5)
mix_ai_path = str(ACCOMP_DIR / 'satie_mix_ai_accomp.wav')
sf.write(mix_ai_path, mix_ai, sr)

print(f"\n💾 믹스 A (원본 반주): {mix_orig_path}")
print(f"💾 믹스 B (AI 반주):   {mix_ai_path}")

### 5-7. 들어봅시다! 🎧

**A**: 원본 반주 — 메트로놈처럼 딱딱합니다. 연주자의 루바토와 맞지 않습니다.  
**B**: AI 반주 — matchmaker가 연주자의 타이밍을 추적해서 반주를 맞췄습니다.

In [ ]:
from IPython.display import Audio, display, HTML

display(HTML("<h4>A. 녹음 + 원본 반주 (메트로놈 타이밍)</h4>"))
display(Audio(mix_orig, rate=sr))

display(HTML("<h4>B. 녹음 + AI 반주 (연주자 타이밍에 맞춤) ✨</h4>"))
display(Audio(mix_ai, rate=sr))

display(HTML("<h4>C. AI 반주만 (비교용)</h4>"))
display(Audio(accomp_warped, rate=sr))

### 핵심 관찰

- **A**에서는 녹음과 반주 사이에 **타이밍 어긋남**이 느껴지나요?
- **B**에서는 반주가 연주자의 루바토를 **따라가는** 느낌이 드나요?
- 이것이 **AI 반주자**의 핵심입니다: 악보 추적(score following) → 타이밍 동기화

---
## 데모 B: 라이브 AI 반주 서버

데모 A는 녹음 파일을 사후 처리했습니다.  
데모 B는 **실시간으로** 같은 일을 합니다:

```
서버 A (연주 시뮬레이터)          서버 B (AI 반주자)
─────────────────────        ─────────────────────
MIDI 멜로디를 오디오로 재생  ──▶  matchmaker로 실시간 추적
                                    ↓
                              현재 악보 위치 파악
                                    ↓
                              반주 음표를 실시간 스케줄링
                                    ↓
                              FluidSynth로 합성 → 스피커
```

수업에서는 서버에서 실행하며, 학생들은 브라우저로 `/stage` 페이지에서 시각화를 봅니다.  
아래 셀에서 서버를 시작하고 테스트할 수 있습니다.

In [ ]:
print("라이브 AI 반주 서버 실행 방법")
print("=" * 50)
print()
print("1. 터미널에서 서버 시작:")
print("   cd /path/to/pianokit")
print("   conda activate pianokit")
print("   python server/accomp_server.py")
print()
print("2. 서버가 시작되면:")
print("   - matchmaker가 Satie 멜로디 MIDI를 오디오로 재생")
print("   - 실시간으로 악보 위치를 추적")
print("   - 반주 음표를 FluidSynth로 합성하여 출력")
print("   - WebSocket으로 /stage 페이지에 위치 전송")
print()
print("3. 브라우저에서 /stage 페이지를 열면:")
print("   - 실시간 악보 위치가 시각화됩니다")
print("   - 반주가 스피커로 나옵니다")
print()
print("⚠️  이 데모는 서버 환경에서만 실행됩니다 (Colab에서는 불가)")

---
## 6. 산출물 저장 — NB05 /stage 연동

정렬 결과와 반주 오디오를 저장합니다.  
NB05의 `/stage` 페이지는 이 데이터를 사용합니다.

In [ ]:
def build_beat_to_time_map(score_midi_path):
    """악보 MIDI에서 beat -> time(초) 변환 테이블 생성."""
    midi = pretty_midi.PrettyMIDI(score_midi_path)
    times = midi.get_beats()
    return [{'beat': float(i), 'time': float(t)} for i, t in enumerate(times)]

for key, p in pieces.items():
    positions = satie_positions if key == 'satie' else prok_positions
    fr = fr_s if key == 'satie' else fr_p
    
    beat_time_map = build_beat_to_time_map(p['score'])
    
    alignment_data = {
        'piece': key,
        'title': p['title'],
        'score_file': p['score'],
        'audio_file': p['audio'],
        'frame_rate': fr,
        'total_frames': len(positions),
        'positions': positions,
        'beat_to_time': beat_time_map,
    }
    
    out_path = ALIGNMENT_DIR / f'{key}_alignment.json'
    out_path.write_text(json.dumps(alignment_data, ensure_ascii=False))
    print(f"💾 {out_path} ({len(positions)} frames, {len(beat_time_map)} beats)")

# 반주 오디오 목록
print(f"\n💾 반주 산출물:")
for f in sorted(ACCOMP_DIR.glob('*.wav')):
    print(f"   {f}")

print("\n✅ 모든 산출물이 저장되었습니다.")
print("→ NB05에서 /stage 페이지와 연결합니다.")

---
## 협업의 핵심

이 노트북에서 체험한 두 가지:

| | 데모 A (오프라인) | 데모 B (라이브) |
|---|---|---|
| **하는 일** | 녹음 + 악보 → 반주 합성 → 믹스 | 실시간 연주 추적 → 즉시 반주 |
| **핵심 기술** | matchmaker 정렬 + FluidSynth | matchmaker 스트리밍 + FluidSynth |
| **체험 포인트** | A/B 비교 청취 | /stage 시각화 동기화 |

### AI 반주자가 보여주는 것

- AI가 음악을 **만드는** 것이 아닙니다
- AI가 내 연주를 **이해하고**, 거기에 맞춰 **반응**합니다
- 연주자는 AI의 반응을 **평가하고 조정**합니다
- 이것이 AI와 연주자의 **진짜 협업**입니다

---

**→ NB05 (`05_stage.ipynb`)**: 모든 산출물을 하나의 무대로 엮습니다.